# LoRA Finetuning

Finetunes a LoRA adapter on StableDiffusion 1.5's UNet.


Runtime: Colab A100


Dataset: Naruto BLIP captions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshaSrirangam/stable-diffusion-rebuilt/blob/main/notebooks/train.ipynb)

## Environment/Setup

In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device) # should be cuda

cpu


In [ ]:
# clone repo
!git clone https://github.com/HarshaSrirangam/stable-diffusion-rebuilt.git
%cd stable-diffusion-rebuilt
!git pull

In [ ]:
# install dependencies (skip torch/numpy to avoid overwriting Colab's preinstalled packages)
%pip install -e . --no-deps -q
%pip install transformers safetensors tqdm accelerate pyyaml datasets -q

In [ ]:
# mount drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# add symlink from colab runs/ -> content/drive/MyDrive/sd-rebuilt/runs/
# runs/ is gitignored and Colab's disk is wiped when the kernel disconnects, so 
# `runs/` is symlinked to a Google Drive folder, which 
# is used to load LoRA checkpoints
!ln -s /content/drive/MyDrive/sd-rebuilt/runs/ runs
!ls -la runs/

In [ ]:
# download SD1.5 checkpoint from HF
#!python scripts/download_data.py

# if HF requests are slow
!mkdir -p /content/stable-diffusion-rebuilt/data/weights
!cp "/content/drive/MyDrive/sd-rebuilt/data/weights/v1-5-pruned-emaonly.safetensors" "/content/stable-diffusion-rebuilt/data/weights/v1-5-pruned-emaonly.safetensors"
!ls -lh data/weights/   # should be ~4.0 GB

## Training

### Config:
- rank: 16
- alpha: 16
- targets: self and cross attention q/k/v/out projections
- batch size: 8
- learning rate: 1e-4
- n_epochs: 10

These configs are written to `configs/lora.yaml`. The `runs/<run_name>` for this run comes from this config. If a run already exists, the `name` field in `config/lora.yaml` should be modified.

In [ ]:
%%writefile configs/lora.yaml
# training run 1
name: lora
pretrained_path: data/weights/v1-5-pruned-emaonly.safetensors
device: cuda

dataset:
  type: HuggingFace
  name: naruto

r: 16
alpha: 16
targets:
  desc: selfcross
  layers: ["q1", "k1", "v1", "proj_out1", "q2", "k2", "v2", "proj_out2"]

n_epochs: 10
batch_size: 8
lr: 0.0001
seed: 42
log_interval: 10

In [ ]:
!python scripts/train_lora.py

## Outputs
- Checkpoints: `runs/<run_name>/checkpoints/checkpoint-<epoch>.pt`
- Loss curve: `runs/<run_name>/losses.json`
- Frozen config: `runs/<run_name>/training_config.yaml`